# Smart Contract HGNN — Training Pipeline

This notebook runs the full HGNN training pipeline for reentrancy vulnerability detection.

**Two modes:**
1. **Full pipeline** (local or Colab with Slither) — processes contracts from scratch
2. **Pre-processed data** (recommended for Colab) — loads a `.pt` file with pre-processed contracts

For GPU training on Colab, pre-process locally first:
```bash
uv run python scripts/preprocess_dataset.py --output data/processed_dataset.pt
```
Then upload `processed_dataset.pt` to Colab.

## 1. Setup

In [ ]:
import os
import sys

# Detect environment
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Clone the repo
    if not os.path.exists("smart-contract-hgnn"):
        !git clone <YOUR_REPO_URL> smart-contract-hgnn
    os.chdir("smart-contract-hgnn")
    
    # Install dependencies
    !pip install torch torch-geometric networkx numpy scikit-learn pandas
    # Only needed if running full pipeline (not pre-processed):
    # !pip install slither-analyzer solc-select

# Add project root to path
PROJECT_ROOT = os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"Project root: {PROJECT_ROOT}")
print(f"Running on Colab: {IN_COLAB}")

In [ ]:
import csv
import logging
import time

import numpy as np
import torch
import torch.nn as nn

from src.model.hgnn import HGNN
from src.hypergraph.features import FEATURE_DIM

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
)
logger = logging.getLogger("notebook")

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Feature dim: {FEATURE_DIM}")

## 2. Load Data

**Option A**: Load pre-processed data (recommended for Colab GPU training)  
**Option B**: Process contracts from scratch (requires Slither + solc)

In [ ]:
# ==============================
# OPTION A: Load pre-processed data
# ==============================
# Pre-process locally first: uv run python scripts/preprocess_dataset.py
# Then upload data/processed_dataset.pt to Colab

PREPROCESSED_PATH = "data/processed_dataset.pt"
USE_PREPROCESSED = os.path.exists(PREPROCESSED_PATH)

if USE_PREPROCESSED:
    print(f"Loading pre-processed data from {PREPROCESSED_PATH}...")
    saved = torch.load(PREPROCESSED_PATH, weights_only=False)
    fold_data = saved["fold_data"]
    metadata = saved["metadata"]
    print(f"Metadata: {metadata}")
    for i, fold in enumerate(fold_data):
        print(f"  Fold {i+1}: {len(fold['train'])} train, {len(fold['val'])} val")
else:
    print("No pre-processed data found. Will process from scratch (Option B).")
    print("To pre-process: uv run python scripts/preprocess_dataset.py")

In [ ]:
# ==============================
# OPTION B: Process from scratch (requires Slither + solc)
# ==============================
# Skip this cell if you loaded pre-processed data above

if not USE_PREPROCESSED:
    from src.evaluation.train import generate_cv_splits, process_contract_list
    
    print("Generating CV splits...")
    folds = generate_cv_splits(n_splits=3, random_state=42)
    
    fold_data = []
    for fold_idx, fold in enumerate(folds):
        print(f"\nProcessing Fold {fold_idx + 1}...")
        start = time.time()
        train_data = process_contract_list(fold["train"])
        val_data = process_contract_list(fold["val"])
        elapsed = time.time() - start
        print(f"  {len(train_data)}/{len(fold['train'])} train, "
              f"{len(val_data)}/{len(fold['val'])} val ({elapsed:.0f}s)")
        fold_data.append({"train": train_data, "val": val_data})

## 3. Hyperparameters

In [ ]:
# ── Hyperparameters ──────────────────────────
HIDDEN_DIM = 64        # HGNN hidden dimension
N_LAYERS = 2           # Number of message passing layers (2, 3, or 4)
LR = 1e-3              # Learning rate
EPOCHS = 50            # Training epochs per fold
DROPOUT = 0.0          # Dropout rate
USE_LAYERNORM = True   # Layer normalization
SEEDS = [42, 0, 1, 2, 3]  # Random seeds for multi-run evaluation

# Class weights for imbalanced dataset (safe=1.0, vulnerable=2.57)
CLASS_WEIGHTS = torch.tensor([1.0, 2.57], dtype=torch.float32).to(device)

print(f"Config: hidden_dim={HIDDEN_DIM}, n_layers={N_LAYERS}, lr={LR}, "
      f"epochs={EPOCHS}, dropout={DROPOUT}, seeds={SEEDS}")

## 4. Training Functions (GPU-enabled)

In [ ]:
def train_epoch_gpu(model, optimizer, loss_fn, train_data, device):
    """Train one epoch with GPU support."""
    model.train()
    total_loss = 0.0
    total_edges = 0

    for contract in train_data:
        X = torch.tensor(contract["X"], dtype=torch.float32).to(device)
        H_inc = torch.tensor(contract["H_inc"], dtype=torch.float32).to(device)
        E = contract["E"]
        node_index = contract["node_index"]
        label = contract["label"]
        n_edges = contract["n_hyperedges"]

        labels = torch.full((n_edges,), label, dtype=torch.long).to(device)

        logits = model.forward_logits(X, H_inc, E, node_index)
        loss = loss_fn(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * n_edges
        total_edges += n_edges

    return total_loss / max(total_edges, 1)


def evaluate_gpu(model, val_data, device):
    """Evaluate model with GPU support."""
    model.eval()
    all_preds = []
    all_labels = []
    predictions = []

    with torch.no_grad():
        for contract in val_data:
            X = torch.tensor(contract["X"], dtype=torch.float32).to(device)
            H_inc = torch.tensor(contract["H_inc"], dtype=torch.float32).to(device)
            E = contract["E"]
            node_index = contract["node_index"]
            label = contract["label"]
            n_edges = contract["n_hyperedges"]

            y_pred = model(X, H_inc, E, node_index)
            pred_labels = y_pred.argmax(dim=1).cpu().tolist()
            true_labels = [label] * n_edges

            all_preds.extend(pred_labels)
            all_labels.extend(true_labels)

            for j, (pred, prob) in enumerate(zip(pred_labels, y_pred.cpu().tolist())):
                predictions.append({
                    "sol_path": os.path.basename(contract["sol_path"]),
                    "hyperedge_idx": j,
                    "true_label": label,
                    "pred_label": pred,
                    "prob_safe": prob[0],
                    "prob_vuln": prob[1],
                })

    # Compute metrics
    tp = sum(1 for p, l in zip(all_preds, all_labels) if p == 1 and l == 1)
    fp = sum(1 for p, l in zip(all_preds, all_labels) if p == 1 and l == 0)
    tn = sum(1 for p, l in zip(all_preds, all_labels) if p == 0 and l == 0)
    fn = sum(1 for p, l in zip(all_preds, all_labels) if p == 0 and l == 1)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0.0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    accuracy = (tp + tn) / len(all_preds) if all_preds else 0.0

    return {
        "precision": precision, "recall": recall, "f1": f1,
        "fnr": fnr, "fpr": fpr, "accuracy": accuracy,
        "tp": tp, "fp": fp, "tn": tn, "fn": fn,
        "n_total": len(all_preds), "predictions": predictions,
    }

## 5. Train

In [ ]:
all_results = []
metric_names = ["precision", "recall", "f1", "fnr", "fpr", "accuracy"]

for seed in SEEDS:
    print(f"\n{'='*60}")
    print(f"Seed {seed}")
    print(f"{'='*60}")

    seed_fold_metrics = []

    for fold_idx in range(3):
        # Set seeds
        torch.manual_seed(seed)
        np.random.seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed(seed)

        train_data = fold_data[fold_idx]["train"]
        val_data = fold_data[fold_idx]["val"]

        # Create model
        model = HGNN(
            in_dim=FEATURE_DIM,
            hidden_dim=HIDDEN_DIM,
            n_layers=N_LAYERS,
            use_layernorm=USE_LAYERNORM,
            dropout=DROPOUT,
        ).to(device)

        optimizer = torch.optim.Adam(model.parameters(), lr=LR)
        loss_fn = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS)

        print(f"\nFold {fold_idx + 1} | train={len(train_data)}, val={len(val_data)}")

        best_f1 = 0.0
        best_state = None
        loss_history = []

        for epoch in range(EPOCHS):
            avg_loss = train_epoch_gpu(model, optimizer, loss_fn, train_data, device)
            loss_history.append(avg_loss)

            if (epoch + 1) % 10 == 0 or epoch == 0 or epoch == EPOCHS - 1:
                metrics = evaluate_gpu(model, val_data, device)
                print(f"  Epoch {epoch+1:3d} | loss={avg_loss:.4f} | "
                      f"P={metrics['precision']:.3f} R={metrics['recall']:.3f} "
                      f"F1={metrics['f1']:.3f} Acc={metrics['accuracy']:.3f}")

                if metrics["f1"] >= best_f1:
                    best_f1 = metrics["f1"]
                    best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        # Final eval
        final_metrics = evaluate_gpu(model, val_data, device)
        final_metrics["loss_history"] = loss_history
        seed_fold_metrics.append(final_metrics)

        # Save best checkpoint
        if best_state is not None:
            os.makedirs("checkpoints", exist_ok=True)
            torch.save(best_state, f"checkpoints/hgnn_fold{fold_idx+1}_seed{seed}.pt")

    all_results.append({"seed": seed, "fold_metrics": seed_fold_metrics})

print("\nTraining complete!")

## 6. Results Summary

In [ ]:
# Aggregate results across all seeds and folds
all_values = {m: [] for m in metric_names}

print(f"\n{'Seed':>6} {'Fold':>5} {'Prec':>8} {'Recall':>8} {'F1':>8} {'FNR':>8} {'FPR':>8} {'Acc':>8}")
print("-" * 65)

for result in all_results:
    seed = result["seed"]
    for fold_idx, fm in enumerate(result["fold_metrics"]):
        print(f"{seed:>6} {fold_idx+1:>5} ", end="")
        for m in metric_names:
            all_values[m].append(fm[m])
            print(f"{fm[m]:>8.4f}", end="")
        print()

print("-" * 65)
print(f"{'Mean':>12} ", end="")
for m in metric_names:
    vals = np.array(all_values[m])
    print(f"{vals.mean():>8.4f}", end="")
print()
print(f"{'Std':>12} ", end="")
for m in metric_names:
    vals = np.array(all_values[m])
    print(f"{vals.std():>8.4f}", end="")
print()

In [ ]:
# Save results to CSV
os.makedirs("results", exist_ok=True)

# Summary
with open("results/cv_summary.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["metric", "mean", "std"])
    for m in metric_names:
        vals = np.array(all_values[m])
        writer.writerow([m, f"{vals.mean():.4f}", f"{vals.std():.4f}"])

# Detailed
with open("results/cv_detailed.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["seed", "fold", *metric_names])
    for result in all_results:
        seed = result["seed"]
        for fold_idx, fm in enumerate(result["fold_metrics"]):
            row = [seed, fold_idx + 1]
            row.extend(f"{fm[m]:.4f}" for m in metric_names)
            writer.writerow(row)

print("Results saved to results/cv_summary.csv and results/cv_detailed.csv")

## 7. Training Loss Curves

In [ ]:
try:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    for fold_idx in range(3):
        ax = axes[fold_idx]
        for result in all_results:
            seed = result["seed"]
            loss_hist = result["fold_metrics"][fold_idx].get("loss_history", [])
            if loss_hist:
                ax.plot(loss_hist, label=f"seed={seed}", alpha=0.7)
        ax.set_title(f"Fold {fold_idx + 1}")
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Loss")
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

    plt.suptitle("Training Loss per Fold")
    plt.tight_layout()
    plt.savefig("results/loss_curves.png", dpi=150)
    plt.show()
    print("Saved to results/loss_curves.png")
except ImportError:
    print("matplotlib not installed — skipping plots")
    print("Install with: pip install matplotlib")

## 8. Hyperparameter Sweep (Optional)

Uncomment and run to try different configurations.

In [ ]:
# # Hyperparameter sweep
# sweep_configs = [
#     {"hidden_dim": 32,  "n_layers": 2, "lr": 1e-3, "dropout": 0.0},
#     {"hidden_dim": 64,  "n_layers": 2, "lr": 1e-3, "dropout": 0.0},
#     {"hidden_dim": 128, "n_layers": 2, "lr": 1e-3, "dropout": 0.0},
#     {"hidden_dim": 64,  "n_layers": 3, "lr": 1e-3, "dropout": 0.0},
#     {"hidden_dim": 64,  "n_layers": 4, "lr": 1e-3, "dropout": 0.0},
#     {"hidden_dim": 64,  "n_layers": 2, "lr": 5e-4, "dropout": 0.0},
#     {"hidden_dim": 64,  "n_layers": 2, "lr": 1e-3, "dropout": 0.2},
#     {"hidden_dim": 64,  "n_layers": 2, "lr": 1e-3, "dropout": 0.5},
# ]
# 
# sweep_results = []
# for cfg in sweep_configs:
#     print(f"\nConfig: {cfg}")
#     seed = 42
#     torch.manual_seed(seed)
#     np.random.seed(seed)
#     
#     f1_scores = []
#     for fold_idx in range(3):
#         model = HGNN(
#             in_dim=FEATURE_DIM,
#             hidden_dim=cfg["hidden_dim"],
#             n_layers=cfg["n_layers"],
#             dropout=cfg["dropout"],
#         ).to(device)
#         optimizer = torch.optim.Adam(model.parameters(), lr=cfg["lr"])
#         loss_fn = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS)
#         
#         for epoch in range(EPOCHS):
#             train_epoch_gpu(model, optimizer, loss_fn, fold_data[fold_idx]["train"], device)
#         
#         metrics = evaluate_gpu(model, fold_data[fold_idx]["val"], device)
#         f1_scores.append(metrics["f1"])
#         print(f"  Fold {fold_idx+1}: F1={metrics['f1']:.4f}")
#     
#     mean_f1 = np.mean(f1_scores)
#     print(f"  Mean F1: {mean_f1:.4f}")
#     sweep_results.append({**cfg, "mean_f1": mean_f1})
# 
# print("\nSweep Results:")
# for r in sorted(sweep_results, key=lambda x: -x["mean_f1"]):
#     print(f"  F1={r['mean_f1']:.4f} | {r}")